# 00 — Data Ingestion v4

End-to-end pipeline from raw OHLCV to all feature parquets.
Each phase **skips automatically** if its output file already exists.
Set `FORCE_REBUILD = True` in cell 1 to force a full rebuild.

| Phase | Output | Source |
|-------|--------|--------|
| A | `data/external/fear_greed_index.parquet` | alternative.me API |
| B | `data/external/coingecko_market_caps.parquet` | CoinGecko API (~1yr) |
| C | `data/external/approx_market_caps.parquet` | OHLCV × circulating supply |
| D | `data/features/BTCUSDT_1h_v3_features.parquet` | Phases A–C + multi-coin OHLCV |
| E | `data/features/BTCUSDT_1h_structural.parquet` | BTCUSDT 1h OHLCV |
| F | `data/raw/BTCUSDT_1h_taker.parquet` | Binance `/api/v3/klines` (all 12 cols) |
| G | `data/features/BTCUSDT_1h_v4_features.parquet` | V1 + V3 + Phases F, H, I |
| H | `data/raw/BTCUSDT_1h_mark_price.parquet` | Binance `/fapi/v1/markPriceKlines` |
| I | `data/raw/BTCUSDT_1h_index_price.parquet` | Binance `/fapi/v1/indexPriceKlines` |
| J | `data/external/sp500_daily.parquet` | yfinance SPY (S&P 500 ETF) |
| K | `data/features/BTCUSDT_1h_microstructure.parquet` | Phase F taker split → causal Roll/Amihud/imbalance/SampEn |

**Prerequisites:** raw OHLCV parquets in `data/raw/` (from `00_data_ingestion_v0.ipynb`).

**v4 change:** adds Phase K — causal microstructure/complexity features (`src/hmats/features/microstructure.py`). Like every other phase it **skips if its output parquet exists** and reuses the Phase F taker file rather than recomputing volume splits.


In [ ]:
from __future__ import annotations

import json
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy.signal as ss

warnings.filterwarnings('ignore')

# ── Repo root ─────────────────────────────────────────────────────────────────
def _repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found')

REPO       = _repo_root()
RAW_DIR    = REPO / 'data' / 'raw'
EXT_DIR    = REPO / 'data' / 'external'
FEAT_DIR   = REPO / 'data' / 'features'
STATIC_DIR = REPO / 'data' / 'static'
EXT_DIR.mkdir(parents=True, exist_ok=True)
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# Set True to re-generate every output even if it already exists
FORCE_REBUILD = False

def _skip(path: Path, label: str) -> bool:
    """Return True (and print a skip message) if the file exists and FORCE_REBUILD is False."""
    if not FORCE_REBUILD and path.exists():
        size = path.stat().st_size / 1e6
        mtime = datetime.fromtimestamp(path.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        print(f'  SKIP  {label}  →  {path.name}  ({size:.1f} MB, modified {mtime})')
        return True
    return False

SYMBOLS_1H = [
    'BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT',
    'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT', 'DOTUSDT', 'LINKUSDT',
]
ALTS = [s for s in SYMBOLS_1H if s != 'BTCUSDT']

COINGECKO_IDS = {
    'BTCUSDT': 'bitcoin', 'ETHUSDT': 'ethereum', 'BNBUSDT': 'binancecoin',
    'XRPUSDT': 'ripple', 'SOLUSDT': 'solana', 'ADAUSDT': 'cardano',
    'DOGEUSDT': 'dogecoin', 'AVAXUSDT': 'avalanche-2', 'DOTUSDT': 'polkadot',
    'LINKUSDT': 'chainlink', 'USDTUSDT': 'tether', 'USDCUSDT': 'usd-coin',
}

print(f'Repo : {REPO}')
print(f'FORCE_REBUILD = {FORCE_REBUILD}')


---
## Phase 0 — Verify Prerequisites

Checks that the required raw OHLCV parquets exist.


In [ ]:
print('Phase 0 — Verifying raw OHLCV files')
missing = []
for sym in SYMBOLS_1H:
    p = RAW_DIR / f'{sym}_1h.parquet'
    if p.exists():
        df = pd.read_parquet(p, columns=['close'])
        print(f'  OK  {sym}_1h.parquet  ({len(df):,} rows)')
    else:
        print(f'  MISSING  {sym}_1h.parquet')
        missing.append(sym)

if missing:
    print(f'\nWARNING: {len(missing)} OHLCV file(s) missing — Phases B/C/D may be incomplete.')
    print('Run 00_data_ingestion_v0.ipynb first to download raw candle data.')
else:
    print('\nAll required OHLCV files present.')


Phase 0 — Verifying raw OHLCV files
  OK  BTCUSDT_1h.parquet  (76,938 rows)
  OK  ETHUSDT_1h.parquet  (76,938 rows)
  OK  BNBUSDT_1h.parquet  (74,995 rows)
  OK  XRPUSDT_1h.parquet  (70,694 rows)
  OK  SOLUSDT_1h.parquet  (50,776 rows)
  OK  ADAUSDT_1h.parquet  (71,106 rows)
  OK  DOGEUSDT_1h.parquet  (60,442 rows)
  OK  AVAXUSDT_1h.parquet  (49,768 rows)
  OK  DOTUSDT_1h.parquet  (50,591 rows)
  OK  LINKUSDT_1h.parquet  (64,524 rows)

All required OHLCV files present.


---
## Phase A — Fear & Greed Index

Source: [alternative.me/fng/](https://alternative.me/fng/) — free, daily, from Feb 2018.  
Output: `data/external/fear_greed_index.parquet`


In [ ]:
FNG_OUTPUT = EXT_DIR / 'fear_greed_index.parquet'

if _skip(FNG_OUTPUT, 'Fear & Greed Index'):
    pass
else:
    print('Phase A — Downloading Fear & Greed Index...')
    try:
        resp = requests.get('https://api.alternative.me/fng/', params={'limit': '0'}, timeout=30)
        resp.raise_for_status()
        fng_data = resp.json()['data']

        fng_df = pd.DataFrame(fng_data)
        fng_df['value'] = fng_df['value'].astype(int)
        fng_df['timestamp'] = pd.to_datetime(fng_df['timestamp'].astype(int), unit='s', utc=True)
        fng_df = fng_df.rename(columns={'timestamp': 'date', 'value_classification': 'classification'})
        fng_df = fng_df[['date', 'value', 'classification']].sort_values('date').reset_index(drop=True)
        fng_df = fng_df.set_index('date')
        fng_df = fng_df.drop(columns=[c for c in ['time_until_update'] if c in fng_df.columns])

        fng_df.to_parquet(FNG_OUTPUT)
        print(f'  Downloaded {len(fng_df)} daily records')
        print(f'  Range: {fng_df.index.min().date()} → {fng_df.index.max().date()}')
        print(f'  Saved: {FNG_OUTPUT}')
    except Exception as e:
        print(f'  ERROR: {e}')


  SKIP  Fear & Greed Index  →  fear_greed_index.parquet  (0.0 MB, modified 2026-05-29 01:07)


---
## Phase B — CoinGecko Market Caps

Source: CoinGecko free tier — last ~365 days of daily market cap + volume.  
Output: `data/external/coingecko_market_caps.parquet`  
Note: 6.5s sleep between requests to respect the free-tier rate limit.


In [ ]:
CG_OUTPUT = EXT_DIR / 'coingecko_market_caps.parquet'

if _skip(CG_OUTPUT, 'CoinGecko market caps'):
    pass
else:
    print('Phase B — Downloading CoinGecko market caps (~90s due to rate limits)...')
    CG_BASE = 'https://api.coingecko.com/api/v3'
    CG_DAYS = 365
    all_records = []

    for symbol, cg_id in COINGECKO_IDS.items():
        print(f'  {cg_id} ({symbol})...', end=' ', flush=True)
        try:
            resp = requests.get(
                f'{CG_BASE}/coins/{cg_id}/market_chart',
                params={'vs_currency': 'usd', 'days': str(CG_DAYS)},
                timeout=30,
            )
            resp.raise_for_status()
            data = resp.json()
            prices = data.get('prices', [])
            mcaps  = data.get('market_caps', [])
            vols   = data.get('total_volumes', [])
            n = min(len(prices), len(mcaps), len(vols))
            for i in range(n):
                ts = pd.Timestamp(prices[i][0], unit='ms', tz='UTC')
                all_records.append({
                    'date': ts, 'symbol': symbol, 'cg_id': cg_id,
                    'price': prices[i][1],
                    'market_cap':   mcaps[i][1] if mcaps[i][1] else np.nan,
                    'total_volume': vols[i][1]  if vols[i][1]  else np.nan,
                })
            print(f'{n} records')
            time.sleep(6.5)
        except Exception as e:
            print(f'ERROR: {e}')
            time.sleep(10)

    if all_records:
        cg_df = pd.DataFrame(all_records)
        cg_df.to_parquet(CG_OUTPUT, index=False)
        print(f'\n  Total: {len(cg_df)} records across {cg_df["symbol"].nunique()} coins')
        print(f'  Range: {cg_df["date"].min().date()} → {cg_df["date"].max().date()}')
        print(f'  Saved: {CG_OUTPUT}')
    else:
        print('  No data downloaded.')


  SKIP  CoinGecko market caps  →  coingecko_market_caps.parquet  (0.1 MB, modified 2026-05-29 01:10)


---
## Phase C — Approximate Historical Market Caps

Extends CoinGecko history back to 2017 by multiplying daily close × circulating supply.  
Circulating supply is a static snapshot from `data/static/crypto_market_caps.csv`.  
Output: `data/external/approx_market_caps.parquet`


In [ ]:
APPROX_OUTPUT = EXT_DIR / 'approx_market_caps.parquet'

if _skip(APPROX_OUTPUT, 'Approximate market caps'):
    pass
else:
    print('Phase C — Computing approximate market caps from OHLCV × supply...')
    SUPPLY_PATH = STATIC_DIR / 'crypto_market_caps.csv'
    if not SUPPLY_PATH.exists():
        print(f'  ERROR: {SUPPLY_PATH} not found — skipping Phase C.')
    else:
        supply_df  = pd.read_csv(SUPPLY_PATH)
        supply_map = dict(zip(supply_df['symbol'], supply_df['circulating_supply']))
        print(f'  Loaded circulating supply for {len(supply_map)} coins')

        records = []
        for symbol in SYMBOLS_1H:
            p = RAW_DIR / f'{symbol}_1h.parquet'
            if not p.exists():
                print(f'  {symbol}: OHLCV missing, skipping')
                continue
            supply = supply_map.get(symbol, np.nan)
            if np.isnan(supply):
                print(f'  {symbol}: no supply data, skipping')
                continue
            df    = pd.read_parquet(p)
            if df.index.tz is None:
                df.index = df.index.tz_localize('UTC')
            daily = df['close'].resample('1D').last().dropna()
            for ts, price in daily.items():
                records.append({
                    'date': ts, 'symbol': symbol,
                    'close_price': price,
                    'circulating_supply': supply,
                    'approx_market_cap': price * supply,
                })
            print(f'  {symbol}: {len(daily)} daily records, supply={supply:,.0f}')

        if records:
            approx_df = pd.DataFrame(records)
            approx_df.to_parquet(APPROX_OUTPUT, index=False)
            print(f'\n  Total: {len(approx_df)} records')
            print(f'  Range: {approx_df["date"].min().date()} → {approx_df["date"].max().date()}')
            print(f'  Saved: {APPROX_OUTPUT}')


  SKIP  Approximate market caps  →  approx_market_caps.parquet  (0.4 MB, modified 2026-05-29 01:10)


---
## Phase D — V3 External Features

Builds 21 features from external data. Requires Phases A–C to be complete.  
Output: `data/features/BTCUSDT_1h_v3_features.parquet`

| Group | Prefix | Features |
|-------|--------|----------|
| Cross-asset | `cross_` | ETH/BTC ratio, altcoin breadth, relative strength, correlation |
| Market structure | `mkt_` | BTC/ETH dominance, total market cap change, stablecoin share |
| Sentiment | `sent_` | Fear & Greed index, 7d MA, 7d change |
| Microstructure | `micro_` | Amihud illiquidity, Kyle lambda, Roll spread, volume clock |
| Enhanced OHLCV | — | Volatility term structure, volatility-normalised momentum |

> Kyle's lambda uses a rolling Python loop (~30s on full history).


In [ ]:
V3_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet'

if _skip(V3_OUTPUT, 'V3 external features'):
    pass
else:
    print('Phase D — Building V3 features...')

    # ── Load base data ────────────────────────────────────────────────────────
    btc = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet')
    btc.index = btc.index.tz_localize(None) if btc.index.tz else btc.index

    alt_closes = {}
    for sym in ALTS:
        p = RAW_DIR / f'{sym}_1h.parquet'
        if p.exists():
            df = pd.read_parquet(p)
            df.index = df.index.tz_localize(None) if df.index.tz else df.index
            alt_closes[sym] = df['close'].reindex(btc.index, method='ffill')

    fng_df = pd.read_parquet(EXT_DIR / 'fear_greed_index.parquet')
    fng_df.index = fng_df.index.tz_localize(None) if fng_df.index.tz else fng_df.index

    approx_mcap = pd.read_parquet(EXT_DIR / 'approx_market_caps.parquet')
    approx_mcap['date'] = pd.to_datetime(approx_mcap['date']).dt.tz_localize(None)

    cg_mcap = pd.read_parquet(EXT_DIR / 'coingecko_market_caps.parquet')
    cg_mcap['date'] = pd.to_datetime(cg_mcap['date']).dt.tz_localize(None)

    print(f'  BTC: {len(btc):,} bars  |  Alts loaded: {list(alt_closes.keys())}')

    v3 = pd.DataFrame(index=btc.index)

    # ── Group 1: Cross-Asset ──────────────────────────────────────────────────
    eth = alt_closes.get('ETHUSDT')
    if eth is not None:
        eth_btc = eth / btc['close']
        v3['cross_eth_btc_ratio']   = eth_btc
        v3['cross_eth_btc_mom_24h'] = eth_btc.pct_change(24)
        v3['cross_eth_btc_mom_72h'] = eth_btc.pct_change(72)

    alt_rets = pd.DataFrame({sym: c.pct_change(24) for sym, c in alt_closes.items()})
    if not alt_rets.empty:
        btc_ret_24h = btc['close'].pct_change(24)
        avg_alt_ret = alt_rets.mean(axis=1)
        v3['cross_altcoin_breadth_24h']   = (alt_rets > 0).mean(axis=1)
        v3['cross_btc_relative_strength'] = btc_ret_24h - avg_alt_ret
        v3['cross_alt_correlation_24h']   = btc_ret_24h.rolling(24).corr(avg_alt_ret)
    print(f'  Group 1 (cross-asset): {sum(1 for c in v3.columns if c.startswith("cross_"))} features')

    # ── Group 2: Market Structure ─────────────────────────────────────────────
    mcap_pivot = approx_mcap.pivot_table(
        index='date', columns='symbol', values='approx_market_cap', aggfunc='last'
    )
    our_coins = [s for s in SYMBOLS_1H if s in mcap_pivot.columns]
    total_mcap = mcap_pivot[our_coins].sum(axis=1)
    btc_dom = mcap_pivot.get('BTCUSDT', pd.Series(dtype=float)) / total_mcap
    eth_dom = mcap_pivot.get('ETHUSDT', pd.Series(dtype=float)) / total_mcap

    cg_stable = cg_mcap[cg_mcap['symbol'].isin(['USDTUSDT', 'USDCUSDT'])]
    if not cg_stable.empty:
        stable_daily  = cg_stable.pivot_table(
            index=cg_stable['date'].dt.normalize(), columns='symbol',
            values='market_cap', aggfunc='last'
        ).sum(axis=1)
        stable_reindx = stable_daily.reindex(mcap_pivot.index, method='ffill')
        stablecoin_pct = stable_reindx / (total_mcap + stable_reindx).replace(0, np.nan)
    else:
        stablecoin_pct = pd.Series(np.nan, index=mcap_pivot.index)

    mkt_daily = pd.DataFrame({
        'mkt_btc_dominance':       btc_dom,
        'mkt_btc_dominance_chg_7d': btc_dom.diff(7),
        'mkt_eth_dominance':       eth_dom,
        'mkt_total_mcap_chg_24h':  total_mcap.pct_change(1),
        'mkt_stablecoin_pct':      stablecoin_pct,
    })
    for col in mkt_daily.columns:
        v3[col] = mkt_daily[col].reindex(btc.index, method='ffill')
    print(f'  Group 2 (market structure): {sum(1 for c in v3.columns if c.startswith("mkt_"))} features')

    # ── Group 3: Sentiment ────────────────────────────────────────────────────
    fng_scaled = fng_df[['value']].rename(columns={'value': 'sent_fear_greed'})
    fng_scaled['sent_fear_greed']       = fng_scaled['sent_fear_greed'] / 100.0
    fng_scaled['sent_fear_greed_ma7']   = fng_scaled['sent_fear_greed'].rolling(7).mean()
    fng_scaled['sent_fear_greed_chg_7d']= fng_scaled['sent_fear_greed'].diff(7)
    for col in fng_scaled.columns:
        v3[col] = fng_scaled[col].reindex(btc.index, method='ffill')
    print(f'  Group 3 (sentiment): {sum(1 for c in v3.columns if c.startswith("sent_"))} features')

    # ── Group 4: Microstructure ───────────────────────────────────────────────
    close      = btc['close']
    volume     = btc['volume']
    dollar_vol = close * volume
    ret        = close.pct_change()
    MICRO_WIN  = 24

    v3['micro_amihud_illiq'] = (
        ret.abs() / dollar_vol.replace(0, np.nan)
    ).rolling(MICRO_WIN).mean()

    # Kyle's lambda: rolling OLS of |Δprice| on volume
    print('  Computing Kyle lambda (~30s)...', flush=True)
    delta_p = close.diff().abs()
    kl = np.full(len(btc), np.nan)
    dp_arr = delta_p.values
    v_arr  = volume.values
    for i in range(MICRO_WIN, len(btc)):
        dp = dp_arr[i - MICRO_WIN:i]
        v  = v_arr[i - MICRO_WIN:i]
        mask = np.isfinite(dp) & np.isfinite(v) & (v > 0)
        if mask.sum() > 5 and v[mask].std() > 0:
            kl[i] = np.polyfit(v[mask], dp[mask], 1)[0]
    v3['micro_kyle_lambda'] = pd.Series(kl, index=btc.index)

    cov_serial = ret.rolling(MICRO_WIN).cov(ret.shift(1))
    v3['micro_roll_spread']   = pd.Series(
        np.where(cov_serial < 0, np.sqrt(-2 * cov_serial), 0.0), index=btc.index
    )
    vol_mean = volume.rolling(MICRO_WIN).mean()
    vol_std  = volume.rolling(MICRO_WIN).std()
    v3['micro_volume_clock'] = vol_std / vol_mean.replace(0, np.nan)
    print(f'  Group 4 (microstructure): {sum(1 for c in v3.columns if c.startswith("micro_"))} features')

    # ── Group 5: Enhanced OHLCV ───────────────────────────────────────────────
    vol_24  = ret.rolling(24).std()
    vol_168 = ret.rolling(168).std()
    v3['vol_term_structure']   = vol_24 / vol_168.replace(0, np.nan)
    v3['mom_normalized_24h']   = close.pct_change(24) / vol_24.replace(0, np.nan)
    v3['mom_normalized_72h']   = close.pct_change(72) / vol_168.replace(0, np.nan)
    print(f'  Group 5 (enhanced OHLCV): {sum(1 for c in v3.columns if c.startswith("vol_term") or c.startswith("mom_norm"))} features')

    v3.to_parquet(V3_OUTPUT)
    print(f'\n  Total V3 features: {v3.shape[1]}')
    print(f'  Date range: {v3.index.min().date()} → {v3.index.max().date()}')
    print(f'  Saved: {V3_OUTPUT}')


  SKIP  V3 external features  →  BTCUSDT_1h_v3_features.parquet  (8.5 MB, modified 2026-05-29 01:19)


---
## Phase E — V2-1h Structural Features

Builds 39 physics-inspired features directly from 1h BTCUSDT OHLCV.  
Output: `data/features/BTCUSDT_1h_structural.parquet`

| Group | Prefix | Features |
|-------|--------|----------|
| Structure | `struct_` | Confirmed swing high/low (±12h minor, ±48h major), proximity flags, wick/body ratios |
| Liquidity | `liq_` | Anchored VWAP (daily/weekly/24h/168h), rolling POC, volume z-score, exhaustion spikes |
| Volatility | `volat_` | ATR 20h/72h, BB width/position, Bollinger-Keltner squeeze, Garman-Klass |
| MTF context | `mtf_` | 4h and daily EMA spread, RSI, composite alignment, session timing |


In [ ]:
STRUCT_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_structural.parquet'

if _skip(STRUCT_OUTPUT, 'V2-1h structural features'):
    pass
else:
    print('Phase E — Building V2-1h structural features...')

    # Window parameters (1h bars)
    SWING_ORDER_S = 12;  SWING_ORDER_L = 48;  NEAR_THRESH = 0.003
    VOC_WIN_S = 24;      VOC_WIN_L = 168;     POC_BINS = 30
    BB_WIN = 20;         ATR_WIN_S = 20;      ATR_WIN_L = 72
    H4_EMA_FAST = 20;    H4_EMA_SLOW = 50;   RSI_PERIOD = 14
    BARS_PER_YEAR = 365.25 * 24
    BURN_IN = 200

    df1h = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet')
    if df1h.index.tz is None:
        df1h.index = df1h.index.tz_localize('UTC')
    df1h = df1h[['open', 'high', 'low', 'close', 'volume']].astype('float64')
    print(f'  Loaded 1h OHLCV: {len(df1h):,} bars  {df1h.index[0].date()} → {df1h.index[-1].date()}')

    # ── Group A: Swing-extrema anchors ────────────────────────────────────────
    def _confirmed_extrema(arr, comparator, order, idx):
        n = len(arr)
        raw_idx = ss.argrelextrema(arr, comparator, order=order)[0]
        prices  = np.full(n, np.nan)
        for ri in raw_idx:
            ci = ri + order
            if ci < n:
                prices[ci] = arr[ri]
        s  = pd.Series(prices, index=idx)
        ff = s.ffill()
        is_new  = s.notna()
        counter = is_new.cumsum()
        age     = is_new.groupby(counter).cumcount().where(counter > 0).astype('float32')
        return ff.astype('float32'), age

    close_arr = df1h['close'].values;  high_arr = df1h['high'].values;  low_arr = df1h['low'].values
    sh_s, sh_s_age = _confirmed_extrema(high_arr, np.greater_equal, SWING_ORDER_S, df1h.index)
    sl_s, sl_s_age = _confirmed_extrema(low_arr,  np.less_equal,    SWING_ORDER_S, df1h.index)
    sh_l, _        = _confirmed_extrema(high_arr, np.greater_equal, SWING_ORDER_L, df1h.index)
    sl_l, _        = _confirmed_extrema(low_arr,  np.less_equal,    SWING_ORDER_L, df1h.index)

    close = df1h['close']
    total_range = (df1h['high'] - df1h['low']).replace(0, np.nan)
    body_top = df1h[['open', 'close']].max(axis=1)
    body_bot = df1h[['open', 'close']].min(axis=1)

    grp_A = pd.DataFrame({
        'struct_dist_swing_high_s': (sh_s - close) / close,
        'struct_dist_swing_low_s':  (close - sl_s) / close,
        'struct_dist_swing_high_l': (sh_l - close) / close,
        'struct_dist_swing_low_l':  (close - sl_l) / close,
        'struct_near_swing_high_s': ((sh_s - close).abs() / close < NEAR_THRESH).astype('float32'),
        'struct_near_swing_low_s':  ((close - sl_s).abs() / close < NEAR_THRESH).astype('float32'),
        'struct_swing_high_age_s':  sh_s_age,
        'struct_swing_low_age_s':   sl_s_age,
        'struct_lower_wick_ratio':  ((body_bot - df1h['low']) / total_range).astype('float32'),
        'struct_upper_wick_ratio':  ((df1h['high'] - body_top) / total_range).astype('float32'),
        'struct_body_ratio':        ((df1h['close'] - df1h['open']).abs() / total_range).astype('float32'),
    }, index=df1h.index)
    print(f'  Group A (structure):   {grp_A.shape[1]} features')

    # ── Group B: Liquidity proxies ────────────────────────────────────────────
    def _anchored_vwap(df, freq):
        tp  = (df['high'] + df['low'] + df['close']) / 3
        pv  = tp * df['volume']
        key = df.index.floor('D') if freq == 'D' else df.index.to_period('W').start_time.tz_localize('UTC')
        return (pv.groupby(key).cumsum() / df['volume'].groupby(key).cumsum().replace(0, np.nan)).astype('float64')

    def _rolling_vwap(df, w):
        tp = (df['high'] + df['low'] + df['close']) / 3
        pv = tp * df['volume']
        return pv.rolling(w, min_periods=1).sum() / df['volume'].rolling(w, min_periods=1).sum()

    def _rolling_poc(close_np, vol_np, window, n_bins):
        n = len(close_np)
        edges   = np.linspace(close_np.min(), close_np.max(), n_bins + 1)
        centers = (edges[:-1] + edges[1:]) / 2
        bar_bin = np.clip(np.digitize(close_np, edges) - 1, 0, n_bins - 1)
        vol_mat = np.zeros((n, n_bins), dtype=np.float32)
        vol_mat[np.arange(n), bar_bin] = vol_np.astype(np.float32)
        cum = np.cumsum(vol_mat, axis=0)
        lagged = np.vstack([np.zeros((window, n_bins), dtype=np.float32), cum[:-window]])
        poc = centers[np.argmax(cum - lagged, axis=1)].astype(np.float32)
        poc[:window] = np.nan
        return poc

    vol_np  = df1h['volume'].to_numpy(np.float64)
    cls_np  = close.to_numpy(np.float64)
    grp_B = pd.DataFrame(index=df1h.index)
    for tag, ser in [
        ('daily',  _anchored_vwap(df1h, 'D')),
        ('weekly', _anchored_vwap(df1h, 'W')),
        (f'{VOC_WIN_S}h', _rolling_vwap(df1h, VOC_WIN_S)),
        (f'{VOC_WIN_L}h', _rolling_vwap(df1h, VOC_WIN_L)),
    ]:
        grp_B[f'liq_vwap_dev_{tag}'] = ((close - ser) / ser).astype('float32')
    for win, tag in [(VOC_WIN_S, f'{VOC_WIN_S}h'), (VOC_WIN_L, f'{VOC_WIN_L}h')]:
        poc = _rolling_poc(cls_np, vol_np, win, POC_BINS)
        grp_B[f'liq_poc_dist_{tag}'] = ((cls_np - poc) / poc).astype('float32')
        vm = df1h['volume'].rolling(win, min_periods=win//2).mean().shift(1)
        vs = df1h['volume'].rolling(win, min_periods=win//2).std().shift(1)
        grp_B[f'liq_vol_z_{tag}'] = ((df1h['volume'] - vm) / vs.replace(0, np.nan)).clip(-5, 5).astype('float32')
    z_l  = grp_B[f'liq_vol_z_{VOC_WIN_L}h']
    body = (df1h['close'] - df1h['open']).astype('float32')
    grp_B['liq_exhaustion_bull'] = ((z_l > 3) & (body > 0)).astype('float32')
    grp_B['liq_exhaustion_bear'] = ((z_l > 3) & (body < 0)).astype('float32')
    print(f'  Group B (liquidity):   {grp_B.shape[1]} features')

    # ── Group C: Volatility ───────────────────────────────────────────────────
    tr = pd.concat([
        df1h['high'] - df1h['low'],
        (df1h['high'] - close.shift(1)).abs(),
        (df1h['low']  - close.shift(1)).abs(),
    ], axis=1).max(axis=1)
    atr_s  = tr.rolling(ATR_WIN_S, min_periods=ATR_WIN_S//2).mean()
    sma20  = close.rolling(BB_WIN, min_periods=BB_WIN//2).mean()
    std20  = close.rolling(BB_WIN, min_periods=BB_WIN//2).std()
    bb_w   = 4 * std20
    kc_w   = 2 * 1.5 * atr_s
    sq     = (bb_w / kc_w.replace(0, np.nan))
    log_hl = np.log(df1h['high'] / df1h['low'].replace(0, np.nan))
    log_co = np.log(close / df1h['open'].replace(0, np.nan))
    gk_bar = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2

    grp_C = pd.DataFrame({
        f'volat_atr_{ATR_WIN_S}_pct': (tr.rolling(ATR_WIN_S, min_periods=ATR_WIN_S//2).mean() / close).astype('float32'),
        f'volat_atr_{ATR_WIN_L}_pct': (tr.rolling(ATR_WIN_L, min_periods=ATR_WIN_L//2).mean() / close).astype('float32'),
        'volat_bb_width_20':          (bb_w / close).astype('float32'),
        'volat_bb_position_20':       ((close - (sma20 - 2*std20)) / bb_w.replace(0, np.nan)).clip(0, 1).astype('float32'),
        'volat_bk_squeeze':           sq.clip(0, 3).astype('float32'),
        'volat_squeeze_on':           (sq < 1.0).astype('float32'),
        f'volat_gk_{ATR_WIN_S}':      np.sqrt(gk_bar.rolling(ATR_WIN_S, min_periods=ATR_WIN_S//2).mean() * BARS_PER_YEAR).astype('float32'),
        f'volat_gk_{ATR_WIN_L}':      np.sqrt(gk_bar.rolling(ATR_WIN_L, min_periods=ATR_WIN_L//2).mean() * BARS_PER_YEAR).astype('float32'),
    }, index=df1h.index)
    print(f'  Group C (volatility):  {grp_C.shape[1]} features')

    # ── Group D: MTF context (4h + daily) ─────────────────────────────────────
    def _wilder_rsi(s, p):
        d  = s.diff()
        eu = d.clip(lower=0).ewm(com=p-1, adjust=False, min_periods=p).mean()
        ed = (-d).clip(lower=0).ewm(com=p-1, adjust=False, min_periods=p).mean()
        return (100 - 100 / (1 + eu / ed.replace(0, np.nan))).astype('float32')

    def _ema_spread(s, fast, slow):
        return ((s.ewm(span=fast, adjust=False).mean() - s.ewm(span=slow, adjust=False).mean()) / s).astype('float32')

    idx1h = df1h.index
    h4c   = df1h['close'].resample('4h', closed='left', label='left').last().dropna()
    h1dc  = df1h['close'].resample('1D', closed='left', label='left').last().dropna()

    h4_clip  = _ema_spread(h4c, H4_EMA_FAST, H4_EMA_SLOW).shift(1).reindex(idx1h, method='ffill').clip(-0.05, 0.05) / 0.05
    h1d_clip = _ema_spread(h1dc, H4_EMA_FAST, H4_EMA_SLOW).shift(1).reindex(idx1h, method='ffill').clip(-0.05, 0.05) / 0.05
    hour = idx1h.hour;  dow = idx1h.dayofweek

    grp_D = pd.DataFrame({
        'mtf_h4_ema_signal':   _ema_spread(h4c, H4_EMA_FAST, H4_EMA_SLOW).shift(1).reindex(idx1h, method='ffill'),
        'mtf_h4_rsi':          (_wilder_rsi(h4c, RSI_PERIOD) / 100).shift(1).reindex(idx1h, method='ffill'),
        'mtf_h4_above_ema50':  (h4c > h4c.ewm(span=H4_EMA_SLOW, adjust=False).mean()).astype('float32').shift(1).reindex(idx1h, method='ffill'),
        'mtf_h1d_ema_signal':  _ema_spread(h1dc, H4_EMA_FAST, H4_EMA_SLOW).shift(1).reindex(idx1h, method='ffill'),
        'mtf_h1d_rsi':         (_wilder_rsi(h1dc, RSI_PERIOD) / 100).shift(1).reindex(idx1h, method='ffill'),
        'mtf_alignment':       (0.4 * h4_clip + 0.6 * h1d_clip).astype('float32'),
        'mtf_session_hour_sin': np.sin(2 * np.pi * hour / 24).astype('float32'),
        'mtf_session_hour_cos': np.cos(2 * np.pi * hour / 24).astype('float32'),
        'mtf_session_dow_sin':  np.sin(2 * np.pi * dow  / 7).astype('float32'),
        'mtf_session_dow_cos':  np.cos(2 * np.pi * dow  / 7).astype('float32'),
    }, index=idx1h)
    print(f'  Group D (MTF):         {grp_D.shape[1]} features')

    # ── Assemble ──────────────────────────────────────────────────────────────
    feat_df = pd.concat([grp_A, grp_B, grp_C, grp_D], axis=1).iloc[BURN_IN:].ffill().dropna()
    assert not feat_df.index.duplicated().any()
    assert feat_df.isna().sum().sum() == 0

    feat_df.to_parquet(STRUCT_OUTPUT)
    print(f'\n  Total features: {feat_df.shape[1]}')
    print(f'  Date range: {feat_df.index[0].date()} → {feat_df.index[-1].date()}')
    print(f'  Saved: {STRUCT_OUTPUT}')


  SKIP  V2-1h structural features  →  BTCUSDT_1h_structural.parquet  (12.6 MB, modified 2026-05-30 22:27)


---
## Phase F — Taker Volume Download

Downloads full Binance 1h klines with microstructure columns (`taker_buy_base_volume`, `num_trades`, `quote_asset_volume`).
Required for **Trade Flow Imbalance (TFI)** features in Phase G.

Output: `data/raw/BTCUSDT_1h_taker.parquet`  
Incremental — subsequent runs only fetch bars newer than the last stored row.


In [ ]:
import sys
sys.path.insert(0, str(REPO / 'src'))
from hmats.data.fetch_taker_volume import fetch_taker_volume

TAKER_OUTPUT = RAW_DIR / 'BTCUSDT_1h_taker.parquet'

if _skip(TAKER_OUTPUT, 'Taker volume (BTCUSDT 1h)'):
    pass
else:
    print('Phase F — Downloading taker volume ...')
    fetch_taker_volume(
        symbol='BTCUSDT',
        interval='1h',
        start='2017-01-01',
        store_dir=str(RAW_DIR),
        verbose=True,
    )
    print('Phase F — done.')


  SKIP  Taker volume (BTCUSDT 1h)  →  BTCUSDT_1h_taker.parquet  (6.2 MB, modified 2026-05-30 23:41)


---
## Phase G — V4 Features

Builds the V4 feature parquet from V1 + V3 base data plus microstructure feeds (Phases F, H, I).

| Feature group | Features | Requires |
|---------------|----------|----------|
| Fractional diff | `fracdiff_close_d{d}` — min stationary d via ADF sweep | V1 |
| Regime | `hurst_24h/72h`, `adf_tstat/pval_Xh`, `bb_width_pct`, `sideways_flag` | V1, V3 |
| TFI | `tfi_pct`, `tfi_z_24/72/168h`, `tfi_ema_12/24` | Phase F |
| Avg trade size | `avg_trade_size`, `avg_trade_size_z24` — institutional block signature | Phase F |
| True VWAP | `true_vwap`, `close_vs_true_vwap` — precise VWAP from quote volume | Phase F |
| Taker price premium | `taker_buy_price`, `taker_sell_price`, `taker_price_premium` — aggressor urgency | Phase F |
| Futures basis | `mark_basis`, `mark_premium_pct`, `mark_premium_z24/168` — funding pressure | Phase H |
| Index deviation | `index_deviation`, `index_deviation_pct`, `index_lead_1h` — lead/lag vs index | Phase I |

Output: `data/features/BTCUSDT_1h_v4_features.parquet`  
Runtime: ~5–10 min (dominated by rolling ADF windows).


In [ ]:
from hmats.data.v4_features import build_v4_features
from hmats.data.fetch_taker_volume import (
    load_taker_volume, load_mark_price_klines, load_index_price_klines,
)

V4_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet'

MARK_OUTPUT  = RAW_DIR / 'BTCUSDT_1h_mark_price.parquet'
INDEX_OUTPUT = RAW_DIR / 'BTCUSDT_1h_index_price.parquet'

if _skip(V4_OUTPUT, 'V4 features'):
    pass
else:
    print('Phase G — Building V4 features ...')
    t0_g = time.time()

    _v1 = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet')
    _v1.index = _v1.index.tz_localize(None) if _v1.index.tz else _v1.index

    _v3 = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet')
    _v3.index = _v3.index.tz_localize(None) if _v3.index.tz else _v3.index

    # ── Optional feeds (only if upstream phases ran) ───────────────────────────
    _taker = None
    if TAKER_OUTPUT.exists():
        _taker = load_taker_volume(store_dir=str(RAW_DIR))
        print(f'  TFI enabled: {len(_taker):,} taker rows')
    else:
        print('  TFI skipped — run Phase F first')

    _mark = None
    if MARK_OUTPUT.exists():
        _mark = load_mark_price_klines(store_dir=str(RAW_DIR))
        print(f'  Mark price enabled: {len(_mark):,} rows')
    else:
        print('  Mark price skipped — run Phase H first')

    _index = None
    if INDEX_OUTPUT.exists():
        _index = load_index_price_klines(store_dir=str(RAW_DIR))
        print(f'  Index price enabled: {len(_index):,} rows')
    else:
        print('  Index price skipped — run Phase I first')

    v4_df = build_v4_features(
        _v1, _v3,
        taker_df=_taker,
        mark_df=_mark,
        index_df=_index,
        hurst_windows=[24, 72],
        adf_windows=[168, 336, 720],
        verbose=True,
    )
    v4_df.to_parquet(V4_OUTPUT)
    print(f'Phase G — done in {(time.time()-t0_g)/60:.1f} min  →  {V4_OUTPUT.name}')
    print(f'  Shape: {v4_df.shape}')
    print(f'  Columns: {list(v4_df.columns)}')


  SKIP  V4 features  →  BTCUSDT_1h_v4_features.parquet  (17.0 MB, modified 2026-05-31 00:03)


---
## Phase H — Futures Mark Price Klines

Downloads Binance Futures **mark price** klines from `/fapi/v1/markPriceKlines`.

Mark price is Binance's manipulation-resistant price — computed as a moving average of
spot prices across multiple exchanges, with clamps to prevent flash-crash distortion.
It drives actual liquidations and funding-rate calculations.

Derived features in Phase G:
- `mark_basis` = mark_close − spot_close (positive → bullish funding pressure)
- `mark_premium_pct` = basis / spot_close (sentiment dial, compare 24h/168h z-score)
- `mark_premium_z24`, `mark_premium_z168` — normalized historical premium

Output: `data/raw/BTCUSDT_1h_mark_price.parquet`  
Available from: **2019-09-01** (Binance Futures launch).  
Incremental — only fetches bars newer than the last cached row.


In [ ]:
from hmats.data.fetch_taker_volume import fetch_mark_price_klines

MARK_OUTPUT = RAW_DIR / 'BTCUSDT_1h_mark_price.parquet'

if _skip(MARK_OUTPUT, 'Mark price klines (BTCUSDT 1h)'):
    pass
else:
    print('Phase H — Downloading mark price klines ...')
    fetch_mark_price_klines(
        symbol='BTCUSDT',
        interval='1h',
        start='2019-09-01',
        store_dir=str(RAW_DIR),
        verbose=True,
    )
    print('Phase H — done.')


  SKIP  Mark price klines (BTCUSDT 1h)  →  BTCUSDT_1h_mark_price.parquet  (2.4 MB, modified 2026-05-31 00:04)


---
## Phase I — Futures Index Price Klines

Downloads Binance Futures **index price** klines from `/fapi/v1/indexPriceKlines`.

The index price is a weighted average of the underlying spot price across multiple exchanges
(Coinbase, Kraken, Bitstamp, etc.). It filters out Binance-specific price moves.

Derived features in Phase G:
- `index_deviation` = index_close − spot_close (absolute lead/lag)
- `index_deviation_pct` = deviation / spot_close
- `index_lead_1h` = index_close.shift(1) − spot_close (can the index predict spot?)

Output: `data/raw/BTCUSDT_1h_index_price.parquet`  
Available from: **2019-09-01** (Binance Futures launch).  
Incremental — only fetches bars newer than the last cached row.


In [ ]:
from hmats.data.fetch_taker_volume import fetch_index_price_klines

INDEX_OUTPUT = RAW_DIR / 'BTCUSDT_1h_index_price.parquet'

if _skip(INDEX_OUTPUT, 'Index price klines (BTCUSDT 1h)'):
    pass
else:
    print('Phase I — Downloading index price klines ...')
    fetch_index_price_klines(
        symbol='BTCUSDT',
        interval='1h',
        start='2019-09-01',
        store_dir=str(RAW_DIR),
        verbose=True,
    )
    print('Phase I — done.')


  SKIP  Index price klines (BTCUSDT 1h)  →  BTCUSDT_1h_index_price.parquet  (2.7 MB, modified 2026-05-31 00:20)


---
## Phase J — S&P 500 Daily Data

Downloads SPY (SPDR S&P 500 ETF) daily OHLCV via `yfinance` as a macro benchmark.
Used in strategy notebooks as a market benchmark alongside the strategy equity curve.

- Ticker: `SPY` (S&P 500 ETF, adjusted closes)
- Frequency: daily, from 2017-01-01
- Output: `data/external/sp500_daily.parquet`

Incremental — if the parquet already exists, only newer bars are appended.

In [ ]:
import yfinance as yf

SP500_OUTPUT = EXT_DIR / 'sp500_daily.parquet'

if _skip(SP500_OUTPUT, 'S&P 500 daily (SPY)'):
    pass
else:
    print('Phase J — Downloading SPY daily data...')
    try:
        _spy = yf.download('SPY', start='2017-01-01', progress=False, auto_adjust=True)
        if _spy.empty:
            raise ValueError('Empty response from yfinance')

        # Flatten MultiIndex columns if present
        if isinstance(_spy.columns, pd.MultiIndex):
            _spy.columns = _spy.columns.get_level_values(0)

        _spy.index = pd.to_datetime(_spy.index).tz_localize(None)
        _spy.index.name = 'date'
        _spy = _spy[['Open', 'High', 'Low', 'Close', 'Volume']].rename(columns=str.lower)

        # Incremental: append only new rows if file already exists
        if SP500_OUTPUT.exists():
            _existing = pd.read_parquet(SP500_OUTPUT)
            _existing.index = pd.to_datetime(_existing.index).tz_localize(None)
            _spy = pd.concat([_existing, _spy])
            _spy = _spy[~_spy.index.duplicated(keep='last')].sort_index()

        _spy.to_parquet(SP500_OUTPUT)
        print(f'  Downloaded {len(_spy):,} daily bars')
        print(f'  Range: {_spy.index.min().date()} → {_spy.index.max().date()}')
        print(f'  Saved: {SP500_OUTPUT}')
    except Exception as e:
        print(f'  ERROR: {e}')
        print('  Ensure yfinance is installed: uv add yfinance')

Phase J — Downloading SPY daily data...
  Downloaded 2,364 daily bars
  Range: 2017-01-03 → 2026-05-29
  Saved: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/external/sp500_daily.parquet


---
## Phase K — Causal Microstructure & Complexity Features

Builds four stationary, strictly-causal features via `hmats.features.microstructure.add_microstructure_features` (every output is `.shift(1)`-ed so bar *t* only sees information through *t-1*).

| Feature | Meaning | Notes |
|---|---|---|
| `roll_measure_50` | Effective-spread / bid-ask bounce intensity | Roll (1984) on **log** price changes (stationary across the 10→100k price range) |
| `amihud_50` | Illiquidity / price impact per unit volume | Amihud (2002), normalised by its trailing 30-day mean |
| `vol_imbalance_50` | Absolute taker buy/sell flow imbalance ∈ [0,1] | Uses the **real** Phase-F taker split, not a tick-rule guess |
| `sampen_48` | Sample Entropy of the 48h log-return path | Bias-free, faster replacement for Approximate Entropy (no O(N²) `rolling.apply`) |

**Reuse, don't recompute.** V3 already ships `micro_amihud_illiq`/`micro_roll_spread` (non-causal, raw-price) and V4 ships `tfi_*` (taker-flow imbalance) and `fracdiff_close_d0.2`. Phase K does **not** touch those; it only adds the new, corrected causal versions plus the genuinely-missing Sample Entropy. Pull whichever set a given model needs at load time.

Output: `data/features/BTCUSDT_1h_microstructure.parquet`  ·  Runtime: <1 min.

In [ ]:
from hmats.features.microstructure import add_microstructure_features

MICRO_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_microstructure.parquet'

if _skip(MICRO_OUTPUT, 'Microstructure & entropy features'):
    pass
elif not (RAW_DIR / 'BTCUSDT_1h_taker.parquet').exists():
    print('Phase K skipped — run Phase F (taker volume) first')
else:
    print('Phase K — Building causal microstructure & entropy features ...')
    t0_k = time.time()

    # Real taker-buy split from Phase F → no tick-rule approximation needed
    _tk = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h_taker.parquet')
    _tk.index = _tk.index.tz_localize(None) if _tk.index.tz else _tk.index
    _src = _tk[['close', 'volume', 'taker_buy_base_volume']].copy()

    _micro = add_microstructure_features(_src, w=50, entropy_w=48)
    _micro = _micro[['roll_measure_50', 'amihud_50', 'vol_imbalance_50', 'sampen_48']]
    _micro.to_parquet(MICRO_OUTPUT)

    print(f'Phase K — done in {(time.time()-t0_k)/60:.1f} min  →  {MICRO_OUTPUT.name}')
    print(f'  Shape: {_micro.shape}  ({_micro.index[0].date()} → {_micro.index[-1].date()})')
    print(f'  Warm-up NaNs: {_micro.isna().sum().to_dict()}')
    print(_micro.describe().T[['min', 'mean', 'max']].round(4).to_string())

---
## Summary


In [ ]:
print('Data pipeline summary')
print('=' * 65)
targets = [
    ('Phase A', EXT_DIR  / 'fear_greed_index.parquet'),
    ('Phase B', EXT_DIR  / 'coingecko_market_caps.parquet'),
    ('Phase C', EXT_DIR  / 'approx_market_caps.parquet'),
    ('Phase D', FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet'),
    ('Phase E', FEAT_DIR / 'BTCUSDT_1h_structural.parquet'),
    ('Phase F', RAW_DIR  / 'BTCUSDT_1h_taker.parquet'),
    ('Phase G', FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet'),
    ('Phase H', RAW_DIR  / 'BTCUSDT_1h_mark_price.parquet'),
    ('Phase I', RAW_DIR  / 'BTCUSDT_1h_index_price.parquet'),
    ('Phase J', EXT_DIR  / 'sp500_daily.parquet'),
    ('Phase K', FEAT_DIR / 'BTCUSDT_1h_microstructure.parquet'),
]
for label, path in targets:
    if path.exists():
        size  = path.stat().st_size / 1e6
        mtime = datetime.fromtimestamp(path.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        print(f'  OK   {label}  {path.name:<50s}  {size:5.1f} MB  {mtime}')
    else:
        print(f'  MISS {label}  {path.name}')


Data pipeline summary
  OK   Phase A  fear_greed_index.parquet                              0.0 MB  2026-05-29 01:07
  OK   Phase B  coingecko_market_caps.parquet                         0.1 MB  2026-05-29 01:10
  OK   Phase C  approx_market_caps.parquet                            0.4 MB  2026-05-29 01:10
  OK   Phase D  BTCUSDT_1h_v3_features.parquet                        8.5 MB  2026-05-29 01:19
  OK   Phase E  BTCUSDT_1h_structural.parquet                        12.6 MB  2026-05-30 22:27
  OK   Phase F  BTCUSDT_1h_taker.parquet                              6.2 MB  2026-05-30 23:41
  OK   Phase G  BTCUSDT_1h_v4_features.parquet                       17.0 MB  2026-05-31 00:03
  OK   Phase H  BTCUSDT_1h_mark_price.parquet                         2.4 MB  2026-05-31 00:04
  OK   Phase I  BTCUSDT_1h_index_price.parquet                        2.7 MB  2026-05-31 00:20
  OK   Phase J  sp500_daily.parquet                                   0.1 MB  2026-05-31 03:42
